In [1]:
# this notebook is for running semantic category classification on convo data

In [1]:
# module imports
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.model_selection import KFold, train_test_split
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import StandardScaler
from scipy.special import gammaln
from scipy.stats import pearsonr, spearmanr
from glob import glob
import os

/scratch/tahaismail424/miniforge3/envs/speech_247_env/lib/python3.11/site-packages/joblib/_multiprocessing_helpers.py:44: UserWarning: [Errno 13] Permission denied.  joblib will operate in serial mode
  warnings.warn("%s.  joblib will operate in serial mode" % (e,))


In [2]:
# set dirs
data_dir = "/mnt/labworlds/Hayden/Hayden_Lab/speech_247/anilu_comparison"

In [3]:
# get patient dirs
patient_dirs = glob(f"{data_dir}/Y*")
patient_dirs = {os.path.basename(dir_):dir_ for dir_ in patient_dirs}

In [4]:
# del YFM for now
del patient_dirs["YFM"]
del patient_dirs["YEZ"]
del patient_dirs["YFA"]

In [5]:
# load patient frs
patient_frs = {}
for pt, dir_ in patient_dirs.items():
    fr_path = f"{dir_}/whisperx_out/whisperx_frs.npy"
    frs = np.load(fr_path)
    patient_frs[pt] = frs

In [6]:
# load transcripts
transcript_dfs = {}
for pt, dir_ in patient_dirs.items():
    transcript_path = glob(f"{dir_}/whisperx_out/audio_transcript_df.csv")[0]
    transcript_df = pd.read_csv(transcript_path, index_col=0)
    transcript_dfs[pt] = transcript_df
    

In [7]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, roc_auc_score, balanced_accuracy_score
from sklearn.preprocessing import label_binarize, LabelEncoder
import matplotlib.pyplot as plt
from scipy.stats import binomtest

In [8]:
# define function for cleaning X and Y
def nan_clean_XY(X, Y):
    """
    1. Drop rows where *all* X values are NaN or *all* Y values are NaN.
    2. Keep everything else (including Y with some NaNs).
    """
    mask_keep = ~np.isnan(X).all(axis=1)
    mask_keep_y = ~(Y == 'nan')
    return X[mask_keep & mask_keep_y], Y[mask_keep & mask_keep_y]


def impute_X_all(Y):
    """
    Impute NaNs in Y using per-channel nanmean from training set only.
    Returns imputed copies of Y_train and Y_test.
    """
    X_imp = X.copy()
    # Compute per-channel mean ignoring NaNs
    channel_means = np.nanmean(X_imp, axis=0)
    # Replace NaN means with 0 if an entire channel is NaN in train
    channel_means = np.where(np.isnan(channel_means), 0.0, channel_means)
    # Fill NaNs
    inds = np.where(np.isnan(X_imp))
    X_imp[inds] = np.take(channel_means, inds[1])
    return X_imp

In [9]:
# ----------------- tiny helpers (standalone) -----------------
def standardize_fit(X):
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, ddof=0, keepdims=True)
    sd = np.where(sd < 1e-8, 1.0, sd)
    return mu, sd

def standardize_apply(X, mu, sd):
    return (X - mu) / sd

def to_device(arr, device):
    if arr is None:
        return None
    if isinstance(arr, np.ndarray):
        if arr.dtype.kind in "iu":
            return torch.tensor(arr, dtype=torch.long, device=device)
        return torch.tensor(arr, dtype=torch.float32, device=device)
    return arr.to(device)

def backtransform_coef(w_std_np, b_std, mu, sd):
    # for standardized X: logits = ( (X - mu)/sd )·w + b
    # => logits = X·(w/sd) + ( b - mu·(w/sd) )
    w_raw = w_std_np / sd.ravel()
    intercept_raw = b_std - float((mu.ravel() @ w_raw))
    return w_raw, intercept_raw

def multinomial_log_likelihood_from_logits(logits_np, y_int):
    # logits: (n, K), y_int: (n,)
    # ll = sum_i log softmax(logits_i)[y_i]
    logits = torch.tensor(logits_np, dtype=torch.float32)
    y = torch.tensor(y_int, dtype=torch.long)
    log_probs = torch.log_softmax(logits, dim=1)
    ll = log_probs[torch.arange(len(y)), y].sum().item()
    return float(ll)

def multinomial_null_loglik(y_int):
    # intercept-only: p_k = class frequency on TRAIN
    counts = np.bincount(y_int)
    p = counts / counts.sum()
    p = np.clip(p, 1e-12, 1.0)
    return float(np.log(p[y_int]).sum())

In [10]:
# load modeling code
from typing import Literal
from joblib import delayed, Parallel
def xgb_manual_cv_early_stop(
    X, y, class_names=None, *,
    params=None,
    test_size=0.2, random_state=42,
    n_iter=40, cv_splits=5, scoring="f1_macro",
    perm_metric: Literal["accuracy","f1_macro","balanced_accuracy"] = "f1_macro",
    n_shuffles: int = 50,
    cv_jobs=1,
    shuf_jobs=1,
    refit_permute: bool = True,
):
    # encode labels
    le = LabelEncoder()
    y_enc = le.fit_transform(np.asarray(y))
    classes = le.classes_
    
    # ----- split
    Xtr, Xte, ytr, yte = train_test_split(
        X, y_enc, test_size=test_size, stratify=y_enc, random_state=random_state
    )
    K = int(np.unique(y).size)

    # ----- base model (GPU)
    base = XGBClassifier(
        objective="multi:softprob",
        num_class=K,
        tree_method="hist",
        predictor="predictor",
        eval_metric="mlogloss",
        random_state=random_state,
        n_estimators=1200,            # we won't use early stopping here
        use_label_encoder=False
    )

    if params is None:
        # ----- search space (good defaults)
        param_dist = {
            "max_depth":        [4,6,8,10],
            "min_child_weight": [1,2,5,10],
            "gamma":            [0, 0.5, 1.0, 2.0],
            "subsample":        [0.7, 0.8, 0.9, 1.0],
            "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
            "reg_lambda":       [0.0, 1.0, 5.0, 10.0],
            "reg_alpha":        [0.0, 0.5, 1.0],
            "learning_rate":    [0.03, 0.05, 0.1],
            "n_estimators":     [400, 800, 1200, 1600],
        }
    
        cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    
        search = RandomizedSearchCV(
            estimator=base,
            param_distributions=param_dist,
            n_iter=n_iter,
            scoring=scoring,
            cv=cv,
            verbose=1,
            n_jobs=cv_jobs,              # keep GPU happy
            refit=True,            # refit on full train with best params
            random_state=random_state
        )
    
        print("starting grid search")
        search.fit(Xtr, ytr)
    
        best_model = search.best_estimator_
    else:
        print("no grid search, fitting model w/ specified params")
        best_model = XGBClassifier(**params)
        best_model.fit(Xtr, ytr)
        
    yhat = best_model.predict(Xte)
    yproba = best_model.predict_proba(Xte)

     # ---------- metrics ----------
    print("calculating metrics")
    acc      = accuracy_score(yte, yhat)
    f1_mac   = f1_score(yte, yhat, average="macro")
    f1_mic   = f1_score(yte, yhat, average="micro")
    bal_acc  = balanced_accuracy_score(yte, yhat)
    cm       = confusion_matrix(yte, yhat)
    report   = classification_report(yte, yhat, target_names=class_names if class_names else None)

    # ROC-AUC (OvR) — optional
    auc_macro = None
    try:
        y_bin = label_binarize(yte, classes=np.arange(K))
        auc_macro = roc_auc_score(y_bin, yproba, multi_class="ovr", average="macro")
    except Exception:
        pass

    # ---------- baselines + binomial test ----------
    # proportional-random baseline accuracy: sum p_c^2 using test-set class proportions
    counts = np.bincount(yte, minlength=K).astype(float)
    p = counts / counts.sum()
    p0_prop = float((p**2).sum())
    p0_major = float(p.max())

    # binomial p-value vs proportional-random baseline (for accuracy)
    # successes = # correct predictions
    k_corr = int((yhat == yte).sum())
    n = int(yte.size)
    binom_p = binomtest(k_corr, n, p0_prop, alternative="greater").pvalue

    # ---------- permutation test (optional) ----------
    perm_scores = None
    perm_pval = None
    rng = np.random.default_rng(random_state)

    def score_metric(y_true, y_pred, which: str):
        if which == "accuracy":
            return accuracy_score(y_true, y_pred)
        elif which == "f1_macro":
            return f1_score(y_true, y_pred, average="macro")
        elif which == "balanced_accuracy":
            return balanced_accuracy_score(y_true, y_pred)
        else:
            raise ValueError("perm_metric must be one of: 'accuracy', 'f1_macro', 'balanced_accuracy'")

    obs_score = score_metric(yte, yhat, perm_metric)

    def _run_perm():
        # shuffle enc labels
        y_enc_shuf = y_enc.copy()
        rng.shuffle(y_enc_shuf)
        Xtr, Xte, ytr, yte = train_test_split(
            X, y_enc_shuf, test_size=test_size, stratify=y_enc, random_state=random_state
        )

        if refit_permute:
            # slow/gold: retrain on the same train set (labels unchanged), evaluate vs shuffled test labels
            m = XGBClassifier(**best_model.get_params())
            m.fit(Xtr, ytr, verbose=False)
            yhat_perm = m.predict(Xte)
        else:
            # fast: reuse the same predictions yhat; only labels are permuted
            yhat_perm = yhat.copy()
            yhat_perm = rng.shuffle(yhat)

        return score_metric(yte, yhat_perm, perm_metric)
        
        
    if n_shuffles and n_shuffles > 0:
        print("running permutations")
        perm_scores = np.empty(n_shuffles, dtype=float)
        if shuf_jobs == 1:
            for s in range(n_shuffles):
                score = _run_perm()
                perm_scores[s] = score
        else:
            scores = Parallel(n_jobs=shuf_jobs, verbose=100, return_as="generator")(
                (delayed(_run_perm)() for r in range(n_shuffles))
            )
            for idx, score in enumerate(scores):
                perm_scores[idx] = score

        # one-sided p-value: fraction of permuted scores >= observed
        perm_pval = (np.sum(perm_scores >= obs_score) + 1) / (n_shuffles + 1)

    # ---------- print summary ----------
    #print(f"\nBest params: {search.best_params_}")
    print(f"Test Accuracy: {acc:.4f}  |  Balanced Acc: {bal_acc:.4f}")
    print(f"Test F1 (macro): {f1_mac:.4f}  |  F1 (micro): {f1_mic:.4f}")
    if auc_macro is not None:
        print(f"ROC-AUC OvR (macro): {auc_macro:.4f}")
    print("\nBaselines:")
    print(f"  Proportional-random acc (∑p_c^2): {p0_prop:.4f}")
    print(f"  Majority-class acc:               {p0_major:.4f}")
    print(f"Binomial p-value vs proportional baseline (accuracy): {binom_p:.3e}")
    if n_shuffles and perm_scores is not None:
        print(f"Permutation test ({perm_metric}): observed={obs_score:.4f}, "
              f"p={perm_pval:.3e}  (n_shuffles={n_shuffles}, refit={refit_permute})")

    results = {
        "y_true": yte,
        "y_pred": yhat,
        "y_proba": yproba,
        "acc": acc,
        "balanced_accuracy": bal_acc,
        "f1_macro": f1_mac,
        "f1_micro": f1_mic,
        "cm": cm,
        "class_report": report,
        "auc_macro_ovr": auc_macro,
        "best_params": best_model.get_params(),
        "cv_best_score": f1_mac,
        "classes": classes,
        "baseline_proportional_acc": p0_prop,
        "baseline_majority_acc": p0_major,
        "binomial_p_vs_proportional": binom_p,
        "perm_metric": perm_metric,
        "perm_scores": perm_scores,
        "perm_pvalue": perm_pval,
    }
    return best_model, results

In [11]:
# define function for class balancing
def class_balance_words(categories, rng: np.random.Generator):
    # remove clusters 10 and -1
    cat = categories[(categories >= 0) & (categories != 10)]
    # get median amount
    median_count = np.round(np.median(cat.value_counts().values)).astype(int)
    to_keep = []
    for cluster_id, counts in cat.value_counts().items():
        cluster_counts = cat[cat == cluster_id]
        # random sample indices
        if len(cluster_counts) > median_count:
            samples = rng.choice(cluster_counts.index, size=median_count, replace=False)
            to_keep.extend(samples)
        else:
            to_keep.extend(cluster_counts.index.values)
    return cat[to_keep], np.array(to_keep)

In [16]:
# now get xgresults for each patient
import dill as pickle
import warnings
warnings.filterwarnings("ignore")
hyperparam_cache = {}
cat_boost_res = {}
n_resamples = 20
for patient in patient_dirs.keys():
    if patient in cat_boost_res:
        continue
    try:
        print("starting patient", patient)
        
        # skip patients for which we already have results
        res_path = f'{data_dir}/{patient}/whisperx_out/scat_xgboost_sampled'
        if len(glob(f"{res_path}/resample*")) == n_resamples:
            print("already have results for", patient)
            continue
        if not os.path.exists(res_path):
            os.mkdir(res_path)
        rng = np.random.default_rng(0)
        frs = patient_frs[patient]
        word_df = transcript_dfs[patient]
        # now run for each resample
        for r in range(n_resamples):
            print("starting resample:", r)
            cat = word_df["semantic_category_pred"].astype(int) - 1
            cat, to_keep = class_balance_words(cat, rng)
            if os.path.exists(f'{res_path}/scat_sampled_resample_{r}_.pkl'):
                print("finished resample alr:", r)
                continue
            cat = cat[to_keep].values
            frs_keep = frs[to_keep]
            _, n_neuron = frs.shape
            # remove nan rows
            mask_keep_x = ~np.isnan(frs_keep).all(axis=1)
            X = frs_keep[mask_keep_x]
            y = cat[mask_keep_x]
            _, n_neuron = frs_keep.shape
            # impute nan neurons w/ average value of that neuron (rounded)
            X = impute_X_all(X)
            # check hyperparam cache
            
            if patient in hyperparam_cache:
                params = hyperparam_cache[patient]
            elif r >= 3:
                param_options = []
                for i in range(3):
                    with open(f'{res_path}/scat_sampled_resample_{i}_.pkl', 'rb') as f:
                        data = pickle.load(f)
                        model, metrics = data
                        param_data = model.get_params()
                        param_options.append(param_data)
                # now get approximate values for our grid based on samples
                max_depth = np.min([param['max_depth'] for param in param_options])
                min_child_weight = np.max([param['min_child_weight'] for param in param_options])
                gamma = np.max([param['gamma'] for param in param_options])
                reg_lambda = np.max([param['reg_lambda'] for param in param_options])
                reg_alpha = np.max([param['reg_alpha'] for param in param_options])
                subsample = np.median([param['subsample'] for param in param_options])
                colsample_bytree = np.median([param['colsample_bytree'] for param in param_options])
                learning_rate = np.median([param['learning_rate'] for param in param_options])
                n_estimators = np.median([param['n_estimators'] for param in param_options])
                # now create our param dict
                params = param_options[0]
                params['max_depth'] = int(max_depth)
                params['min_child_weight'] = int(min_child_weight)
                params['gamma'] = float(gamma)
                params['reg_lambda'] = float(reg_lambda)
                params['reg_alpha'] = float(reg_alpha)
                params['subsample'] = float(subsample)
                params['colsample_bytree'] = float(colsample_bytree)
                params['learning_rate'] = float(learning_rate)
                params['n_estimators'] = int(n_estimators)
                params['tree_method'] = 'hist'
                params['predictor'] = 'predictor'
                hyperparam_cache[patient] = params
                
            else:
                params = None
            results = xgb_manual_cv_early_stop(X, y, cv_splits=4, cv_jobs=25, params=params, random_state=r*42, shuf_jobs=25)
            cat_boost_res[f"patient_{r}"] = results
            # make sure to save seed and indices
            results[1]["seed"] = r*42
            results[1]["sampled_idx"] = to_keep
            with open(f'{res_path}/scat_sampled_resample_{r}_.pkl', 'wb') as f:
                pickle.dump(results, f)
    except Exception as e:
        print(e)

starting patient YFK
starting resample: 0
finished resample alr: 0
starting resample: 1
finished resample alr: 1
starting resample: 2
finished resample alr: 2
starting resample: 3
finished resample alr: 3
starting resample: 4
finished resample alr: 4
starting resample: 5
finished resample alr: 5
starting resample: 6
finished resample alr: 6
starting resample: 7
finished resample alr: 7
starting resample: 8
finished resample alr: 8
starting resample: 9
finished resample alr: 9
starting resample: 10
finished resample alr: 10
starting resample: 11
finished resample alr: 11
starting resample: 12
finished resample alr: 12
starting resample: 13
finished resample alr: 13
starting resample: 14
finished resample alr: 14
starting resample: 15
finished resample alr: 15
starting resample: 16
finished resample alr: 16
starting resample: 17
finished resample alr: 17
starting resample: 18
finished resample alr: 18
starting resample: 19
finished resample alr: 19
starting patient YFI
starting resample: